In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def estimate_blur(image, threshold=200):
    laplacian = cv2.Laplacian(image, cv2.CV_64F)
    variance = laplacian.var()
    return variance, variance < threshold  

def estimate_noise(image):
    blurred = cv2.GaussianBlur(image, (5, 5), 1.5)
    diff = cv2.absdiff(image, blurred)
    noise_level = np.mean(diff)
    return noise_level, noise_level > 10  

def estimate_contrast(image):
    std_dev = np.std(image)
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    non_zero = np.where(hist > 0)[0]
    if len(non_zero) > 0:
        spread = non_zero[-1] - non_zero[0]
    else:
        spread = 0
    
    low_contrast = (std_dev < 40) or (spread < 150)
    return std_dev, spread, low_contrast

def dynamic_preprocess(image):
    
    processed = image.copy()
    steps_applied = []
    
    blur_value, is_blurry = estimate_blur(image)

    
    if is_blurry:
        kernel = np.array([[-1,-1,-1],
                          [-1, 9,-1],
                          [-1,-1,-1]])
        processed = cv2.filter2D(processed, -1, kernel)
        steps_applied.append("Sharpening")

    
    noise_level, is_noisy = estimate_noise(processed)


    if is_noisy:
  
        if noise_level > 20:
            kernel_size = (7, 7)
        elif noise_level > 15:
            kernel_size = (5, 5)
        else:
            kernel_size = (3, 3)
        
        
        processed = cv2.GaussianBlur(processed, kernel_size, 1.0)
        steps_applied.append(f"Gaussian blur {kernel_size}")

    
    
    std_dev, spread, low_contrast = estimate_contrast(processed)

    
    if low_contrast:
        if is_noisy or is_blurry:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            processed = clahe.apply(processed)
            steps_applied.append("CLAHE equalization")
        else:
            processed = cv2.equalizeHist(processed)
            steps_applied.append("Global histogram equalization")
        
    
    return processed, steps_applied


    

count = 0
while True:
    path = "C:/Users/USER/Desktop/image processing project/image processing video/raveling/extracted/"+ str(count) +".jpg"
    img = cv2.imread(path, 0)
    
    
    if img is None:
        print("No frame")
        break
    else:
        processed_img, steps = dynamic_preprocess(img)
        
        cv2.imshow("processed",processed_img)
        cv2.imshow("original",img)
        if cv2.waitKey(100)==ord('q'):
            break
        
    count = count + 5

cv2.destroyAllWindows()